# 07 · Data profiling

A formal profile of every important column in the processed tables: type,
count, unique values, missingness, range, quantiles, spread, IQR outliers
and duplicates. The output is written to `reports/data_profile.md`.

The EDA notebooks explained *what* the data means. This notebook records
*what it looks like* column by column, so that later changes (a new data
release, a schema drift) can be detected by re-running it and diffing the
report.

One idea carries over from the EDA: event columns are sparse **by design**.
A column-level missing rate over all events is mostly meaningless (a
pass length is "missing" for every shot). So events are profiled twice:
once overall, and once **per event type**, where sparse columns become
dense and real gaps become visible.

In [1]:
import polars as pl

from barca import config
from barca.data import profile, store

pl.Config.set_tbl_rows(60)
pl.Config.set_tbl_width_chars(190)
pl.Config.set_tbl_cols(20)
pl.Config.set_tbl_hide_dataframe_shape(True)
pl.Config.set_float_precision(3)

matches = store.matches()
events = store.events().collect()
appearances = store.appearances()
results = store.results()
print({"matches": matches.height, "events": events.height, "appearances": appearances.height, "results": results.height})

{'matches': 869, 'events': 3140830, 'appearances': 29711, 'results': 10044}


## 1. Matches

In [2]:
match_cols = ["match_id", "date", "competition", "season", "home", "away", "home_score", "away_score",
              "match_week", "stadium", "referee", "home_manager", "away_manager", "barca_era"]
p_matches = profile.profile_frame(matches, match_cols)
p_matches

column,dtype,count,unique,missing,missing_pct,min,p01,q1,median,mean,q3,p99,max,std,outliers_iqr,top,top_freq
str,str,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,str,i64
"""match_id""","""Int64""",869,869,0,0.000,9575.000,9661.000,69268.000,267432.000,1755991.832,3825691.000,3825899.000,3825908.000,1835673.921,0,null,null
"""date""","""Date""",869,609,0,0.000,null,null,null,null,null,null,null,null,null,null,null,null
"""competition""","""String""",869,2,0,0.000,null,null,null,null,null,null,null,null,null,null,"""La Liga""",866
"""season""","""String""",869,17,0,0.000,null,null,null,null,null,null,null,null,null,null,"""2015/2016""",380
"""home""","""String""",869,42,0,0.000,null,null,null,null,null,null,null,null,null,null,"""Barcelona""",266
"""away""","""String""",869,40,0,0.000,null,null,null,null,null,null,null,null,null,null,"""Barcelona""",261
"""home_score""","""Int16""",869,10,0,0.000,0.000,0.000,1.000,2.000,1.817,3.000,6.000,10.000,1.602,7,null,null
"""away_score""","""Int16""",869,9,0,0.000,0.000,0.000,0.000,1.000,1.333,2.000,6.000,8.000,1.354,11,null,null
"""match_week""","""Int16""",869,38,0,0.000,1.000,1.000,10.000,20.000,19.367,29.000,38.000,38.000,10.881,0,null,null


In [3]:
print("duplicate match ids:", profile.duplicates(matches, ["match_id"]))
print("duplicate (date, home, away):", profile.duplicates(matches, ["date", "home", "away"]))

duplicate match ids: 0
duplicate (date, home, away): 0


Keys and scores are complete. Missingness is confined to descriptive
metadata: the referee (109 matches) and the managers (about 40 matches per
side, resolved for Barcelona through the tenure table, D-011). Scores span
0-10 for a side.

## 2. Events: overall

In [4]:
event_cols = ["match_id", "index", "period", "minute", "second", "elapsed_s", "type", "team_id", "player_id",
              "x", "y", "end_x", "end_y", "end_z", "duration", "outcome", "subtype", "pass_length",
              "shot_xg", "under_pressure", "pass_complete"]
p_events = profile.profile_frame(events, event_cols)
p_events

column,dtype,count,unique,missing,missing_pct,min,p01,q1,median,mean,q3,p99,max,std,outliers_iqr,top,top_freq,true_pct
str,str,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,str,i64,f64
"""match_id""","""Int64""",3140830,869,0,0.000,9575.000,9650.000,69265.000,267183.000,1676550.666,3825677.000,3825900.000,3825908.000,1822917.464,0,null,null,null
"""index""","""Int32""",3140830,4806,0,0.000,1.000,37.000,904.000,1808.000,1827.129,2711.000,3957.000,4806.000,1076.547,0,null,null,null
"""period""","""Int8""",3140830,2,0,0.000,1.000,1.000,1.000,1.000,1.496,2.000,2.000,2.000,0.500,0,null,null,null
"""minute""","""Int16""",3140830,99,0,0.000,0.000,0.000,21.000,45.000,44.557,67.000,92.000,98.000,26.881,0,null,null,null
"""second""","""Int16""",3140830,60,0,0.000,0.000,0.000,14.000,29.000,29.301,44.000,59.000,59.000,17.382,0,null,null,null
"""elapsed_s""","""Float64""",3140830,6050,0,0.000,0.000,32.000,1297.000,2726.000,2732.570,4131.000,5597.000,6201.000,1639.416,0,null,null,null
"""type""","""String""",3140830,35,0,0.000,null,null,null,null,null,null,null,null,null,null,"""Pass""",881588,null
"""team_id""","""Int32""",3140830,43,0,0.000,39.000,207.000,215.000,217.000,283.110,220.000,1049.000,1230.000,212.719,567059,null,null,null
"""player_id""","""Int32""",3129710,2133,11120,0.350,2948.000,3236.000,5503.000,6735.000,11464.309,20055.000,27526.000,220761.000,9385.978,7034,null,null,null


In [5]:
print("duplicate event ids:", profile.duplicates(events, ["event_id"]))
print("duplicate (match_id, index):", profile.duplicates(events, ["match_id", "index"]))

duplicate event ids: 0
duplicate (match_id, index): 0


Read in isolation, the overall profile would suggest that half the dataset
is broken: `end_x` is 48% missing, `outcome` 86%, `shot_xg` 99%. None of
that is a defect. The per-type view shows it:

## 3. Events: per type

In [6]:
per_type = pl.concat(
    [
        profile.profile_by_type(events, "Pass", ["x", "y", "end_x", "end_y", "pass_length", "pass_angle", "recipient_id", "outcome", "duration"]),
        profile.profile_by_type(events, "Carry", ["x", "y", "end_x", "end_y", "duration"]),
        profile.profile_by_type(events, "Shot", ["x", "y", "end_x", "end_y", "end_z", "shot_xg", "outcome", "shot_freeze_frame_n"]),
        profile.profile_by_type(events, "Pressure", ["x", "y", "duration"]),
        profile.profile_by_type(events, "Duel", ["x", "y", "subtype", "outcome"]),
    ],
    how="diagonal_relaxed",
).select("event_type", "column", "dtype", "count", "missing_pct", "min", "median", "mean", "max", "std", "outliers_iqr")
per_type

event_type,column,dtype,count,missing_pct,min,median,mean,max,std,outliers_iqr
str,str,str,i64,f64,f64,f64,f64,f64,f64,i64
"""Pass""","""x""","""Float32""",881588,0.000,0.100,59.600,59.324,120.000,27.162,0
"""Pass""","""y""","""Float32""",881588,0.000,0.100,40.100,40.200,80.000,23.417,0
"""Pass""","""end_x""","""Float32""",881588,0.000,0.100,66.400,66.343,120.000,26.499,0
"""Pass""","""end_y""","""Float32""",881588,0.000,0.100,40.100,40.079,80.000,22.990,0
"""Pass""","""pass_length""","""Float32""",881588,0.000,0.000,16.896,21.045,119.946,14.579,54355
"""Pass""","""pass_angle""","""Float32""",881588,0.000,-3.139,-0.008,0.006,3.142,1.543,0
"""Pass""","""recipient_id""","""Int32""",816836,7.340,2948.000,6707.000,11166.594,220761.000,9275.279,2509
"""Pass""","""outcome""","""String""",180230,79.560,null,null,null,null,null,null
"""Pass""","""duration""","""Float32""",881588,0.000,0.000,1.344,1.546,65.175,0.900,44918


Per type, the expected fields are complete: every pass has start and end
locations and a length, every carry an end, every shot an xG, an outcome
and a freeze frame. The remaining "missing" values are the ones that are
null *by definition*: pass outcome (complete), shot `end_z` (the height at
the goal line, absent for 27% of shots, mostly blocked or wide ones) and
duel outcome for aerials (every "Aerial Lost" is, by name, lost).

Notable distributions:

* **Pass length**: median about 17 yards, 99th percentile about 71 yards
  (goal kicks, switches). The IQR rule flags the long passes as "outliers", but they are
  real long balls, not errors.
* **Shot xG**: heavily right-skewed. The median shot is worth about 0.06
  goals, and every penalty is assigned the same xG (0.78).
* **Duration** has a long tail for carries (a player dribbling a long way);
  zero-duration events are instantaneous actions.

This is the general lesson of the profile: **IQR "outliers" in football data
are usually the interesting events** (a 60-yard switch, a 0.9 xG tap-in),
so outliers are *reported*, never removed. Only values that are physically
impossible count as errors, and those are encoded as validation rules
(notebook 08).

## 4. Appearances (lineups and minutes)

In [7]:
app_cols = ["match_id", "team_id", "player_id", "display_name", "jersey_number", "country", "started", "played",
            "minutes", "match_minutes", "position_group", "on_minute", "off_minute", "n_yellow", "n_red"]
p_app = profile.profile_frame(appearances, app_cols)
p_app

column,dtype,count,unique,missing,missing_pct,min,p01,q1,median,mean,q3,p99,max,std,outliers_iqr,top,top_freq,true_pct
str,str,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,str,i64,f64
"""match_id""","""Int64""",29711,869,0,0.000,9575.000,9650.000,69288.000,303524.000,1878721.798,3825702.000,3825900.000,3825908.000,1842093.825,0,null,null,null
"""team_id""","""Int32""",29711,43,0,0.000,39.000,206.000,214.000,217.000,301.687,221.000,1217.000,1230.000,238.073,5157,null,null,null
"""player_id""","""Int32""",29711,2413,0,0.000,2948.000,3240.000,6378.000,6866.000,13856.319,22819.000,40915.000,402733.000,16700.802,186,null,null,null
"""display_name""","""String""",29711,2363,0,0.000,null,null,null,null,null,null,null,null,null,null,"""Lionel Messi""",522,null
"""jersey_number""","""Int16""",29711,50,0,0.000,1.000,1.000,7.000,13.000,13.396,20.000,33.000,58.000,8.128,70,null,null,null
"""country""","""String""",29704,97,7,0.020,null,null,null,null,null,null,null,null,null,null,"""Spain""",16385,null
"""started""","""Boolean""",29711,2,0,0.000,null,null,null,null,null,null,null,null,null,null,null,null,64.350
"""played""","""Boolean""",29711,2,0,0.000,null,null,null,null,null,null,null,null,null,null,null,null,81.570
"""minutes""","""Float64""",29711,5199,0,0.000,0.000,0.000,19.067,82.917,60.647,94.033,98.267,103.350,38.836,0,null,null,null


In [8]:
print("duplicate (match, player):", profile.duplicates(appearances, ["match_id", "player_id"]))
appearances.group_by("started", "played").agg(pl.len().alias("rows"), pl.col("minutes").mean().alias("mean_minutes"))

duplicate (match, player): 0


started,played,rows,mean_minutes
bool,bool,u32,f64
true,true,19118,87.447
false,false,5477,0.000
false,true,5116,25.423


Three kinds of row: starters (≈ 90+ minutes unless substituted),
substitutes who came on, and unused substitutes (0 minutes, `played =
False`, recorded from 2015/16 onwards). `position_group` is missing only
for unused substitutes, who have no position.

## 5. Results (football-data)

In [9]:
res_cols = ["season", "date", "home", "away", "home_goals", "away_goals", "home_goals_ht", "home_shots",
            "home_sot", "home_corners", "home_yellow", "home_red", "odds_b365_h", "odds_b365_d", "odds_b365_a",
            "odds_psc_h", "odds_avg_h"]
p_res = profile.profile_frame(results, res_cols)
p_res

column,dtype,count,unique,missing,missing_pct,top,top_freq,min,p01,q1,median,mean,q3,p99,max,std,outliers_iqr
str,str,i64,i64,i64,f64,str,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64
"""season""","""String""",10044,26,0,0.000,"""1995/1996""",462,null,null,null,null,null,null,null,null,null,null
"""date""","""Date""",10044,2636,0,0.000,null,null,null,null,null,null,null,null,null,null,null,null
"""home""","""String""",10044,47,0,0.000,"""Athletic Club""",498,null,null,null,null,null,null,null,null,null,null
"""away""","""String""",10044,47,0,0.000,"""Valencia""",498,null,null,null,null,null,null,null,null,null,null
"""home_goals""","""Int16""",10044,11,0,0.000,null,null,0.000,0.000,1.000,1.000,1.562,2.000,5.000,10.000,1.308,839
"""away_goals""","""Int16""",10044,9,0,0.000,null,null,0.000,0.000,0.000,1.000,1.119,2.000,5.000,8.000,1.116,29
"""home_goals_ht""","""Int16""",10042,8,2,0.020,null,null,0.000,0.000,0.000,0.000,0.691,1.000,3.000,6.000,0.842,342
"""home_shots""","""Int16""",6080,38,3964,39.470,null,null,1.000,4.000,10.000,13.000,13.712,17.000,28.000,37.000,5.026,69
"""home_sot""","""Int16""",6080,21,3964,39.470,null,null,0.000,0.000,3.000,5.000,4.930,6.000,13.000,20.000,2.626,211


Scores are complete for all 10,044 fixtures. Match statistics are missing
exactly for the seasons before 2005/06 and closing odds before 2012/13,
the coverage boundaries found in notebook 01, not scattered gaps.

## 6. Write the profiling report

In [10]:
sections = [
    ("Matches", p_matches), ("Events (overall)", p_events), ("Events (per type)", per_type),
    ("Appearances", p_app), ("Results (football-data)", p_res),
]
keep = ["column", "dtype", "count", "unique", "missing", "missing_pct", "min", "median", "mean", "max", "std",
        "outliers_iqr", "top", "true_pct"]
lines = [
    "# Data profile",
    "",
    "Generated by `notebooks/07_data_profiling.ipynb` from the processed tables. "
    "Event columns are sparse by design, so events are also profiled per type.",
    "",
]
for title, frame in sections:
    cols = [c for c in (["event_type"] if "event_type" in frame.columns else []) + keep if c in frame.columns]
    lines += [f"## {title}", "", profile.to_markdown(frame.select(cols)), ""]
(config.REPORTS / "data_profile.md").write_text("\n".join(lines))
print("written reports/data_profile.md")

written reports/data_profile.md


## Findings

1. **Keys are clean**: no duplicate matches, events, fixtures or
   player-match rows.
2. **Missingness is structural**: overall missing rates in the event table
   are an artefact of the wide schema. Per type, expected fields are
   complete.
3. **Outliers are football, not errors**: long passes, high-xG chances and
   long carries are kept. Only physically impossible values are errors.
4. **Coverage gaps are boundaries, not holes**: football-data stats before
   2005/06 and closing odds before 2012/13.

Next: turning these observations into enforceable rules (`08_data_quality`).